# Sprint 5 — Pré-treinamento em dados não rotulados

Projeto Integrador — Construção de um Large Language Model (LLM) From Scratch

Baseado no Capítulo 5 (*Pretraining on unlabeled data*) do livro **Build a Large Language Model (From Scratch)**, de Sebastian Raschka.

Este notebook reúne, em sequência, os códigos da sprint (`src/Sprint 5/5,1.py` a `5,5.py`): a avaliação do modelo com cross-entropy e perplexidade (Entregável 2), a rotina de treinamento (Entregável 3), as estratégias de geração de texto (Entregável 5), o salvamento e carregamento dos pesos (Entregável 6) e, como extra, o carregamento dos pesos pré-treinados da OpenAI. O Entregável 4 (experimentos de treinamento) está em `src/Sprint 5/Experimentos.py`, com os resultados documentados em `relatório/Sprint 5/`.

## Preparação do ambiente

Mesma organização das sprints anteriores: o texto vem de `data/the-verdict.txt`. O notebook deve ser aberto de dentro da pasta `notebooks/`. Os pesos treinados são salvos em `src/Sprint 5/` (arquivos `.pth`, fora do Git, como no repositório do livro).

In [ ]:
import json
import math
import time
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import tiktoken
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator

BASE_DIR = Path.cwd().resolve().parents[0]  # ajuste se necessario: pasta raiz do repositorio
PASTA_SPRINT = BASE_DIR / "src" / "Sprint 5"
PASTA_RESULTADOS = BASE_DIR / "relatório" / "Sprint 5" / "resultados"

print("PyTorch:", torch.__version__)

## Componentes das sprints anteriores

O treinamento usa, sem nenhuma alteração, os componentes construídos nas Sprints 2 a 4: o `GPTDatasetV1` e o `create_dataloader_v1` (Sprint 2), a `MultiHeadAttention` (Sprint 3) e o `TransformerBlock` com o `GPTModel` (Sprint 4). A única mudança está na configuração: o `context_length` cai de 1024 para 256 tokens, como no livro, para o treino caber em um computador comum.

In [ ]:
class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})

        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1: i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]


def create_dataloader_v1(txt, batch_size=4, max_length=256,
                         stride=128, shuffle=True, drop_last=True,
                         num_workers=0):
    tokenizer = tiktoken.get_encoding("gpt2")
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)
    dataloader = DataLoader(
        dataset, batch_size=batch_size, shuffle=shuffle,
        drop_last=drop_last, num_workers=num_workers
    )
    return dataloader


class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert (d_out % num_heads == 0), "d_out must be divisible by num_heads"

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length), diagonal=1)
        )

    def forward(self, x):
        b, num_tokens, d_in = x.shape
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)

        keys = keys.transpose(1, 2)
        queries = queries.transpose(1, 2)
        values = values.transpose(1, 2)

        attn_scores = queries @ keys.transpose(2, 3)
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]
        attn_scores.masked_fill_(mask_bool, -torch.inf)

        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        context_vec = (attn_weights @ values).transpose(1, 2)
        context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)
        context_vec = self.out_proj(context_vec)
        return context_vec


class LayerNorm(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.eps = 1e-5
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim))

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        norm_x = (x - mean) / torch.sqrt(var + self.eps)
        return self.scale * norm_x + self.shift


class GELU(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x):
        return 0.5 * x * (1 + torch.tanh(
            torch.sqrt(torch.tensor(2.0 / torch.pi)) *
            (x + 0.044715 * torch.pow(x, 3))
        ))


class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(cfg["emb_dim"], 4 * cfg["emb_dim"]),
            GELU(),
            nn.Linear(4 * cfg["emb_dim"], cfg["emb_dim"]),
        )

    def forward(self, x):
        return self.layers(x)


class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.att = MultiHeadAttention(
            d_in=cfg["emb_dim"], d_out=cfg["emb_dim"],
            context_length=cfg["context_length"], num_heads=cfg["n_heads"],
            dropout=cfg["drop_rate"], qkv_bias=cfg["qkv_bias"],
        )
        self.ff = FeedForward(cfg)
        self.norm1 = LayerNorm(cfg["emb_dim"])
        self.norm2 = LayerNorm(cfg["emb_dim"])
        self.drop_shortcut = nn.Dropout(cfg["drop_rate"])

    def forward(self, x):
        shortcut = x
        x = self.norm1(x)
        x = self.att(x)
        x = self.drop_shortcut(x)
        x = x + shortcut

        shortcut = x
        x = self.norm2(x)
        x = self.ff(x)
        x = self.drop_shortcut(x)
        x = x + shortcut
        return x


class GPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.drop_emb = nn.Dropout(cfg["drop_rate"])

        self.trf_blocks = nn.Sequential(
            *[TransformerBlock(cfg) for _ in range(cfg["n_layers"])]
        )

        self.final_norm = LayerNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False)

    def forward(self, in_idx):
        batch_size, seq_len = in_idx.shape
        tok_embeds = self.tok_emb(in_idx)
        pos_embeds = self.pos_emb(torch.arange(seq_len, device=in_idx.device))
        x = tok_embeds + pos_embeds
        x = self.drop_emb(x)
        x = self.trf_blocks(x)
        x = self.final_norm(x)
        logits = self.out_head(x)
        return logits


def generate_text_simple(model, idx, max_new_tokens, context_size):
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -context_size:]
        with torch.no_grad():
            logits = model(idx_cond)
        logits = logits[:, -1, :]
        probas = torch.softmax(logits, dim=-1)
        idx_next = torch.argmax(probas, dim=-1, keepdim=True)
        idx = torch.cat((idx, idx_next), dim=1)
    return idx


# Contexto reduzido de 1024 para 256 tokens, como no Capitulo 5 do livro
GPT_CONFIG_124M = {
    "vocab_size": 50257,
    "context_length": 256,
    "emb_dim": 768,
    "n_heads": 12,
    "n_layers": 12,
    "drop_rate": 0.1,
    "qkv_bias": False,
}

## 5.1 — Avaliação de modelos generativos de texto

### 5.1.1 — Usando o GPT para gerar texto

Duas funções auxiliares fazem a ponte entre texto e tensores: `text_to_token_ids` (texto → IDs com dimensão de lote) e `token_ids_to_text` (o caminho inverso). Também definimos aqui as funções de carregamento do texto, de divisão em treino/validação e de cálculo da perda, usadas no resto do notebook.

In [ ]:
def escolher_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


def carregar_texto():
    with open(BASE_DIR / "data" / "the-verdict.txt", "r", encoding="utf-8") as f:
        return f.read()


def criar_loaders(text_data, batch_size, context_length, train_ratio=0.90):
    split_idx = int(train_ratio * len(text_data))
    train_data = text_data[:split_idx]
    val_data = text_data[split_idx:]

    train_loader = create_dataloader_v1(
        train_data, batch_size=batch_size, max_length=context_length,
        stride=context_length, drop_last=True, shuffle=True, num_workers=0,
    )
    val_loader = create_dataloader_v1(
        val_data, batch_size=batch_size, max_length=context_length,
        stride=context_length, drop_last=False, shuffle=False, num_workers=0,
    )
    return train_loader, val_loader


def text_to_token_ids(text, tokenizer):
    encoded = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
    encoded_tensor = torch.tensor(encoded).unsqueeze(0)
    return encoded_tensor


def token_ids_to_text(token_ids, tokenizer):
    flat = token_ids.squeeze(0)
    return tokenizer.decode(flat.tolist())


def calc_loss_batch(input_batch, target_batch, model, device):
    input_batch, target_batch = input_batch.to(device), target_batch.to(device)
    logits = model(input_batch)
    loss = torch.nn.functional.cross_entropy(logits.flatten(0, 1), target_batch.flatten())
    return loss


def calc_loss_loader(data_loader, model, device, num_batches=None):
    total_loss = 0.
    if len(data_loader) == 0:
        return float("nan")
    elif num_batches is None:
        num_batches = len(data_loader)
    else:
        num_batches = min(num_batches, len(data_loader))
    for i, (input_batch, target_batch) in enumerate(data_loader):
        if i < num_batches:
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            total_loss += loss.item()
        else:
            break
    return total_loss / num_batches

### Aplicando — o modelo sem treino

In [ ]:
device = escolher_device()
print("Device:", device)
tokenizer = tiktoken.get_encoding("gpt2")

torch.manual_seed(123)
model = GPTModel(GPT_CONFIG_124M)
model.eval()

token_ids = generate_text_simple(
    model=model,
    idx=text_to_token_ids("Every effort moves you", tokenizer),
    max_new_tokens=10,
    context_size=GPT_CONFIG_124M["context_length"],
)
print("Texto gerado:", token_ids_to_text(token_ids, tokenizer))

### 5.1.2 — Cross-entropy e perplexidade

Para dois exemplos curtos, os alvos são as próprias entradas deslocadas em uma posição. A perda é calculada passo a passo: softmax sobre os logits → probabilidade do token correto → logaritmo → média → troca de sinal. O resultado é conferido com `torch.nn.functional.cross_entropy`, e a perplexidade é a exponencial da perda.

In [ ]:
inputs = torch.tensor([[16833, 3626, 6100],    # "every effort moves"
                       [40,    1107, 588]])    # "I really like"
targets = torch.tensor([[3626, 6100, 345],     # " effort moves you"
                        [1107,  588, 11311]])  # " really like chocolate"

with torch.no_grad():
    logits = model(inputs)
probas = torch.softmax(logits, dim=-1)
print("Shape das probabilidades:", probas.shape)

token_ids = torch.argmax(probas, dim=-1, keepdim=True)
print("Alvos    lote 1:", token_ids_to_text(targets[0], tokenizer))
print("Previsto lote 1:", token_ids_to_text(token_ids[0].flatten(), tokenizer))

In [ ]:
target_probas_1 = probas[0, [0, 1, 2], targets[0]]
target_probas_2 = probas[1, [0, 1, 2], targets[1]]
print("Texto 1:", target_probas_1)
print("Texto 2:", target_probas_2)
print("Chute uniforme (1/50257):", 1 / GPT_CONFIG_124M["vocab_size"])

log_probas = torch.log(torch.cat((target_probas_1, target_probas_2)))
avg_log_probas = torch.mean(log_probas)
neg_avg_log_probas = avg_log_probas * -1
print("Log-probabilidades:", log_probas)
print("Media:", avg_log_probas)
print("Perda manual (media negativa):", neg_avg_log_probas)

In [ ]:
logits_flat = logits.flatten(0, 1)
targets_flat = targets.flatten()
print("Logits achatados:", logits_flat.shape, "| Alvos achatados:", targets_flat.shape)

loss = torch.nn.functional.cross_entropy(logits_flat, targets_flat)
print("cross_entropy do PyTorch:", loss)
print("Perplexidade:", torch.exp(loss))
print("ln(50257) =", math.log(50257))

### 5.1.3 — Perdas dos conjuntos de treino e de validação

O conto *The Verdict* é dividido em 90% para treino e 10% para validação. Os dois loaders usam janelas de 256 tokens sem sobreposição (`stride = 256`) e lotes de 2 sequências.

In [ ]:
text_data = carregar_texto()
total_characters = len(text_data)
total_tokens = len(tokenizer.encode(text_data))
print("Caracteres:", total_characters)
print("Tokens:", total_tokens)

torch.manual_seed(123)
train_loader, val_loader = criar_loaders(
    text_data, batch_size=2, context_length=GPT_CONFIG_124M["context_length"])

print("Loader de treino:")
for x, y in train_loader:
    print(" ", x.shape, y.shape)
print("Loader de validacao:")
for x, y in val_loader:
    print(" ", x.shape, y.shape)

In [ ]:
model.to(device)
torch.manual_seed(123)
with torch.no_grad():
    train_loss = calc_loss_loader(train_loader, model, device)
    val_loss = calc_loss_loader(val_loader, model, device)

print("Perda de treino:", train_loss, "| perplexidade:", math.exp(train_loss))
print("Perda de validacao:", val_loss, "| perplexidade:", math.exp(val_loss))

## 5.2 — Treinando o LLM (Entregável 3)

A rotina `train_model_simple` percorre as épocas e, em cada uma, os lotes de treino: zera os gradientes, calcula a perda, retropropaga (`loss.backward()`) e atualiza os pesos (`optimizer.step()`). A cada `eval_freq` passos, `evaluate_model` mede as perdas de treino e validação em modo `eval()`; ao fim de cada época, uma amostra de texto é gerada. Como adaptação nossa, a função também guarda um histórico por época, com as perdas sobre os conjuntos inteiros, usado nos experimentos.

In [ ]:
def evaluate_model(model, train_loader, val_loader, device, eval_iter):
    model.eval()
    with torch.no_grad():
        train_loss = calc_loss_loader(train_loader, model, device, num_batches=eval_iter)
        val_loss = calc_loss_loader(val_loader, model, device, num_batches=eval_iter)
    model.train()
    return train_loss, val_loss


def generate_and_print_sample(model, tokenizer, device, start_context):
    model.eval()
    context_size = model.pos_emb.weight.shape[0]
    encoded = text_to_token_ids(start_context, tokenizer).to(device)
    with torch.no_grad():
        token_ids = generate_text_simple(
            model=model, idx=encoded,
            max_new_tokens=50, context_size=context_size
        )
    decoded_text = token_ids_to_text(token_ids, tokenizer)
    print(decoded_text.replace("\n", " "))
    model.train()
    return decoded_text


def train_model_simple(model, train_loader, val_loader, optimizer, device, num_epochs,
                       eval_freq, eval_iter, start_context, tokenizer,
                       historico=None, loaders_completos=None):
    # Adaptacao nossa: historico guarda perdas e amostra de cada epoca, medidas em loaders sem embaralhar
    train_losses, val_losses, track_tokens_seen = [], [], []
    tokens_seen, global_step = 0, -1
    inicio = time.perf_counter()

    for epoch in range(num_epochs):
        model.train()

        for input_batch, target_batch in train_loader:
            optimizer.zero_grad()
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            loss.backward()
            optimizer.step()
            tokens_seen += input_batch.numel()
            global_step += 1

            if global_step % eval_freq == 0:
                train_loss, val_loss = evaluate_model(
                    model, train_loader, val_loader, device, eval_iter)
                train_losses.append(train_loss)
                val_losses.append(val_loss)
                track_tokens_seen.append(tokens_seen)
                print(f"Ep {epoch+1} (Step {global_step:06d}): "
                      f"Train loss {train_loss:.3f}, Val loss {val_loss:.3f}")

        amostra = generate_and_print_sample(
            model, tokenizer, device, start_context
        )

        if historico is not None:
            loader_treino, loader_val = loaders_completos or (train_loader, val_loader)
            perda_treino, perda_val = evaluate_model(
                model, loader_treino, loader_val, device, eval_iter=None)
            historico.append({
                "epoca": epoch + 1,
                "passos": global_step + 1,
                "tokens_vistos": tokens_seen,
                "train_loss": perda_treino,
                "val_loss": perda_val,
                "tempo_s": time.perf_counter() - inicio,
                "amostra": amostra,
            })

    return train_losses, val_losses, track_tokens_seen


def plot_losses(epochs_seen, tokens_seen, train_losses, val_losses, caminho):
    fig, ax1 = plt.subplots(figsize=(5, 3))

    ax1.plot(epochs_seen, train_losses, label="Perda de treino")
    ax1.plot(epochs_seen, val_losses, linestyle="-.", label="Perda de validação")
    ax1.set_xlabel("Épocas")
    ax1.set_ylabel("Perda")
    ax1.legend(loc="upper right")
    ax1.xaxis.set_major_locator(MaxNLocator(integer=True))

    ax2 = ax1.twiny()
    ax2.plot(tokens_seen, train_losses, alpha=0)
    ax2.set_xlabel("Tokens vistos")

    fig.tight_layout()
    plt.savefig(caminho, dpi=150)
    plt.show()

### Aplicando — 10 épocas com AdamW (configuração do livro)

In [ ]:
CONFIG_TREINO = {
    "learning_rate": 0.0004,
    "weight_decay": 0.1,
    "num_epochs": 10,
    "batch_size": 2,
    "eval_freq": 5,
    "eval_iter": 5,
    "train_ratio": 0.90,
    "seed": 123,
    "start_context": "Every effort moves you",
}

torch.manual_seed(123)
model = GPTModel(GPT_CONFIG_124M)
model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG_TREINO["learning_rate"],
                              weight_decay=CONFIG_TREINO["weight_decay"])

inicio = time.perf_counter()
train_losses, val_losses, tokens_seen = train_model_simple(
    model, train_loader, val_loader, optimizer, device,
    num_epochs=CONFIG_TREINO["num_epochs"], eval_freq=CONFIG_TREINO["eval_freq"],
    eval_iter=CONFIG_TREINO["eval_iter"], start_context=CONFIG_TREINO["start_context"],
    tokenizer=tokenizer,
)
print(f"Treinamento concluido em {(time.perf_counter() - inicio) / 60:.2f} minutos.")

In [ ]:
PASTA_RESULTADOS.mkdir(parents=True, exist_ok=True)
epochs_tensor = torch.linspace(0, CONFIG_TREINO["num_epochs"], len(train_losses))
plot_losses(epochs_tensor, tokens_seen, train_losses, val_losses,
            PASTA_RESULTADOS / "5_2_perdas.png")

In [ ]:
model.to("cpu")
model.eval()
token_ids = generate_text_simple(
    model=model,
    idx=text_to_token_ids("Every effort moves you", tokenizer),
    max_new_tokens=25,
    context_size=GPT_CONFIG_124M["context_length"],
)
print("Texto gerado:\n", token_ids_to_text(token_ids, tokenizer))

## 5.3 — Estratégias de decodificação (Entregável 5)

### 5.3.1 — Temperatura

Exemplo do livro com um vocabulário de 9 palavras e logits fixos para o próximo token depois de *"every effort moves you"*. O greedy pega sempre o maior valor; a amostragem com `torch.multinomial` sorteia de acordo com as probabilidades. A temperatura divide os logits antes da softmax: T < 1 concentra a distribuição e T > 1 a achata.

In [ ]:
vocab = {
    "closer": 0, "every": 1, "effort": 2, "forward": 3, "inches": 4,
    "moves": 5, "pizza": 6, "toward": 7, "you": 8,
}
inverse_vocab = {v: k for k, v in vocab.items()}
next_token_logits = torch.tensor([4.51, 0.89, -1.90, 6.75, 1.63, -1.62, -1.89, 6.28, 1.79])

probas = torch.softmax(next_token_logits, dim=0)
print("Greedy:", inverse_vocab[torch.argmax(probas).item()])
torch.manual_seed(123)
print("Amostragem:", inverse_vocab[torch.multinomial(probas, num_samples=1).item()])

In [ ]:
def softmax_with_temperature(logits, temperature):
    scaled_logits = logits / temperature
    return torch.softmax(scaled_logits, dim=0)


def print_sampled_tokens(probas, inverse_vocab, n=1000):
    torch.manual_seed(123)
    sample = [torch.multinomial(probas, num_samples=1).item() for _ in range(n)]
    sampled_ids = torch.bincount(torch.tensor(sample), minlength=len(probas))
    contagem = {}
    for i, freq in enumerate(sampled_ids):
        print(f"  {freq} x {inverse_vocab[i]}")
        contagem[inverse_vocab[i]] = int(freq)
    return contagem

def entropia(probas):
    p = probas[probas > 0]
    return float(-(p * torch.log(p)).sum())


temperatures = [1, 0.1, 5]
scaled_probas = [softmax_with_temperature(next_token_logits, T) for T in temperatures]
for T, p in zip(temperatures, scaled_probas):
    print(f"\n1000 amostras com T={T} (entropia {entropia(p):.3f}):")
    print_sampled_tokens(p, inverse_vocab)

In [ ]:
x = torch.arange(len(vocab))
bar_width = 0.15
fig, ax = plt.subplots(figsize=(5, 3))
for i, T in enumerate(temperatures):
    ax.bar(x + i * bar_width, scaled_probas[i], bar_width, label=f"Temperatura = {T}")
ax.set_ylabel("Probabilidade")
ax.set_xticks(x)
ax.set_xticklabels(vocab.keys(), rotation=90)
ax.legend()
plt.tight_layout()
plt.show()

### 5.3.2 — Top-k

O top-k mantém apenas os k maiores logits, coloca −∞ nos demais e só então aplica a softmax. Com k = 3, apenas *forward*, *toward* e *closer* podem ser sorteadas.

In [ ]:
top_k = 3
top_logits, top_pos = torch.topk(next_token_logits, top_k)
print("Top logits:", top_logits)
print("Top posicoes:", top_pos)

new_logits = torch.where(
    condition=next_token_logits < top_logits[-1],
    input=torch.tensor(float("-inf")),
    other=next_token_logits,
)
print(new_logits)
print(torch.softmax(new_logits, dim=0))

### 5.3.3 — Função `generate` com temperatura e top-k

A `generate` substitui a `generate_text_simple` da Sprint 4. Com `temperature=0` ela continua sendo greedy; com temperatura > 0 ela sorteia o próximo token, opcionalmente só entre os `top_k` mais prováveis. O parâmetro `eos_id` interrompe a geração quando o token de parada é escolhido.

In [ ]:
def generate(model, idx, max_new_tokens, context_size, temperature=0.0, top_k=None, eos_id=None):
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -context_size:]
        with torch.no_grad():
            logits = model(idx_cond)
        logits = logits[:, -1, :]

        if top_k is not None:
            top_logits, _ = torch.topk(logits, top_k)
            min_val = top_logits[:, -1]
            logits = torch.where(
                logits < min_val,
                torch.tensor(float("-inf")).to(logits.device),
                logits
            )

        if temperature > 0.0:
            logits = logits / temperature
            logits = logits - logits.max(dim=-1, keepdim=True).values
            probs = torch.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
        else:
            idx_next = torch.argmax(logits, dim=-1, keepdim=True)

        if eos_id is not None and idx_next.item() == eos_id:
            break

        idx = torch.cat((idx, idx_next), dim=1)

    return idx

### Aplicando — mesmo contexto, modelo sem treino × modelo treinado

In [ ]:
# Adaptacao nossa: diversidade, repeticao e maior trecho copiado do the-verdict
def metricas_texto(ids_novos, ids_corpus_str):
    n = len(ids_novos)
    unicos = len(set(ids_novos)) / n if n else 0.0
    bigramas = list(zip(ids_novos, ids_novos[1:]))
    repetidos = 1 - len(set(bigramas)) / len(bigramas) if bigramas else 0.0
    maior = 0
    for i in range(n):
        j = i + maior + 1
        while j <= n and ("," + ",".join(map(str, ids_novos[i:j])) + ",") in ids_corpus_str:
            maior = j - i
            j += 1
    return {
        "tokens_unicos_pct": 100 * unicos,
        "bigramas_repetidos_pct": 100 * repetidos,
        "maior_trecho_copiado": maior,
    }


ids_corpus_str = "," + ",".join(map(str, tokenizer.encode(text_data))) + ","
contexto = "Every effort moves you"
n_prompt = len(tokenizer.encode(contexto))

torch.manual_seed(123)
model_sem_treino = GPTModel(GPT_CONFIG_124M)
model_sem_treino.eval()

estrategias = [
    ("Greedy (T=0)", {"temperature": 0.0, "top_k": None}),
    ("Amostragem pura (T=1)", {"temperature": 1.0, "top_k": None}),
    ("Temperatura baixa (T=0,5)", {"temperature": 0.5, "top_k": None}),
    ("Temperatura alta (T=1,5)", {"temperature": 1.5, "top_k": None}),
    ("Top-k=5 (T=1)", {"temperature": 1.0, "top_k": 5}),
    ("Livro: top-k=25, T=1,4", {"temperature": 1.4, "top_k": 25}),
]

for nome_modelo, m in [("SEM treino", model_sem_treino), ("TREINADO", model)]:
    print(f"\n===== Modelo {nome_modelo} =====")
    for nome, params in estrategias:
        torch.manual_seed(123)
        ids = generate(model=m, idx=text_to_token_ids(contexto, tokenizer), max_new_tokens=30,
                       context_size=GPT_CONFIG_124M["context_length"], **params)
        met = metricas_texto(ids[0, n_prompt:].tolist(), ids_corpus_str)
        print(f"[{nome}] unicos {met['tokens_unicos_pct']:.0f}% | bigramas repetidos "
              f"{met['bigramas_repetidos_pct']:.0f}% | maior trecho copiado {met['maior_trecho_copiado']}")
        print("  ", repr(token_ids_to_text(ids, tokenizer)))

## 5.4 — Salvando e carregando os pesos (Entregável 6)

O `state_dict` guarda os tensores de cada camada. Salvamos só os pesos (`model.pth`) e, para poder retomar o treino, os pesos junto com o estado do AdamW e as configurações usadas (`model_and_optimizer.pth`). Os arquivos ficam em `src/Sprint 5/` e não vão para o Git (cerca de 650 MB e 1,9 GB).

In [ ]:
PASTA_SPRINT.mkdir(parents=True, exist_ok=True)
torch.save(model.state_dict(), PASTA_SPRINT / "model.pth")
torch.save({
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "gpt_config": GPT_CONFIG_124M,
    "config_treino": CONFIG_TREINO,
}, PASTA_SPRINT / "model_and_optimizer.pth")

for nome in ["model.pth", "model_and_optimizer.pth"]:
    print(nome, f"{(PASTA_SPRINT / nome).stat().st_size / 1024**2:.1f} MB")

### Aplicando — carregando em um modelo novo e conferindo

In [ ]:
model_carregado = GPTModel(GPT_CONFIG_124M)
model_carregado.load_state_dict(torch.load(PASTA_SPRINT / "model.pth", map_location="cpu", weights_only=True))
model_carregado.eval()

iguais = all(torch.equal(a, b) for a, b in zip(model.state_dict().values(),
                                                model_carregado.state_dict().values()))
print("Todos os tensores iguais:", iguais)

for nome, m in [("original", model), ("carregado", model_carregado)]:
    ids = generate(model=m, idx=text_to_token_ids("Every effort moves you", tokenizer),
                   max_new_tokens=25, context_size=GPT_CONFIG_124M["context_length"])
    print(nome, "->", repr(token_ids_to_text(ids, tokenizer)))

### Aplicando — retomando o treino por mais 1 época (exercício 5.4)

In [ ]:
checkpoint = torch.load(PASTA_SPRINT / "model_and_optimizer.pth", map_location=device, weights_only=True)

model_retomado = GPTModel(checkpoint["gpt_config"])
model_retomado.load_state_dict(checkpoint["model_state_dict"])
model_retomado.to(device)
optimizer_retomado = torch.optim.AdamW(model_retomado.parameters(), lr=0.0004, weight_decay=0.1)
optimizer_retomado.load_state_dict(checkpoint["optimizer_state_dict"])
model_retomado.train()

torch.manual_seed(123)
train_model_simple(
    model_retomado, train_loader, val_loader, optimizer_retomado, device,
    num_epochs=1, eval_freq=5, eval_iter=5,
    start_context="Every effort moves you", tokenizer=tokenizer,
)

## 5.5 — Pesos pré-treinados da OpenAI (extra)

O `GPTModel` do projeto tem a mesma arquitetura do GPT-2, então pode receber os pesos publicados pela OpenAI. Usamos o arquivo já convertido para PyTorch pelo autor do livro, a alternativa que o próprio repositório indica para Windows, porque o caminho original depende do TensorFlow. O GPT-2 original usa contexto de 1024 e `qkv_bias=True`.

In [ ]:
import urllib.request

NEW_CONFIG = GPT_CONFIG_124M.copy()
NEW_CONFIG.update({"emb_dim": 768, "n_layers": 12, "n_heads": 12,
                   "context_length": 1024, "qkv_bias": True})

file_name = "gpt2-small-124M.pth"
caminho = PASTA_SPRINT / file_name
url = f"https://huggingface.co/rasbt/gpt2-from-scratch-pytorch/resolve/main/{file_name}"
if not caminho.exists():
    urllib.request.urlretrieve(url, caminho)

gpt = GPTModel(NEW_CONFIG)
gpt.load_state_dict(torch.load(caminho, map_location="cpu", weights_only=True))
gpt.eval()

torch.manual_seed(123)
ids = generate(model=gpt, idx=text_to_token_ids("Every effort moves you", tokenizer),
               max_new_tokens=25, context_size=NEW_CONFIG["context_length"], top_k=50, temperature=1.5)
print(token_ids_to_text(ids, tokenizer))

### Aplicando — perdas do GPT-2 da OpenAI no the-verdict (exercício 5.5)

In [ ]:
gpt.to(device)
torch.manual_seed(123)
with torch.no_grad():
    print("GPT-2 OpenAI -> treino:", calc_loss_loader(train_loader, gpt, device),
          "| validacao:", calc_loss_loader(val_loader, gpt, device))

## Entregável 4 — Experimentos de treinamento

Os experimentos variando o número de épocas, a taxa de aprendizado e o dropout (valores do grid de hiperparâmetros do repositório do livro) estão em `src/Sprint 5/Experimentos.py`. Os resultados, as tabelas comparativas e a análise estão em `relatório/Sprint 5/Sprint5_Experimentacao.pdf` e `relatório/Sprint 5/Sprint5_Análise.pdf`.